## GSAT trend patterns

In [1]:
# In[1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
# %%
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprocess

In [2]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster()

In [3]:
def func_mk(x):
    """
    Mann-Kendall test for trend
    """
    results = data_process.mk_test(x)
    slope = results[0]
    p_val = results[1]
    return slope, p_val

In [4]:
# load data
input_observation = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIGS5_S6/'

HadCRUT5_MMLE = xr.open_dataset(input_observation + 'OBS_SAT_anomaly_partition_wrt_MMLE_ENS.nc')

In [5]:
HadCRUT5_MMLE

<xarray.Dataset>
Dimensions:               (lat: 90, lon: 180, year: 173)
Coordinates:
  * lat                   (lat) float64 -89.0 -87.0 -85.0 ... 85.0 87.0 89.0
  * lon                   (lon) float64 0.0 2.0 4.0 6.0 ... 354.0 356.0 358.0
  * year                  (year) int64 1850 1851 1852 1853 ... 2020 2021 2022
Data variables:
    slope                 (lat, lon) float64 ...
    intercept             (lat, lon) float64 ...
    forced_signal         (year, lat, lon) float64 ...
    internal_variability  (year, lat, lon) float64 ...

In [6]:
HadCRUT5_MMLE['forced_signal']

<xarray.DataArray 'forced_signal' (year: 173, lat: 90, lon: 180)>
[2802600 values with dtype=float64]
Coordinates:
  * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
  * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
  * year     (year) int64 1850 1851 1852 1853 1854 ... 2018 2019 2020 2021 2022

### Plotting with the Robinson Projections

In [ ]:
plt.rcParams['figure.figsize'] = (8, 10)
plt.rcParams['font.size'] = 16
# plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.major.right'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.bottom'] = True
plt.rcParams['savefig.transparent'] = True

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm, ListedColormap

def plot_trend_with_significance(trend_data, lats, lons, p_values, GMST_p_values=None, levels=None, extend=None, cmap=None, 
                                 title="", ax=None, show_xticks=False, show_yticks=False):
    """
    Plot the trend spatial pattern using Robinson projection with significance overlaid.

    Parameters:
    - trend_data: 2D numpy array with the trend values.
    - lats, lons: 1D arrays of latitudes and longitudes.
    - p_values: 2D array with p-values for each grid point.
    - GMST_p_values: 2D array with GMST p-values for each grid point.
    - title: Title for the plot.
    - ax: Existing axis to plot on. If None, a new axis will be created.
    - show_xticks, show_yticks: Boolean flags to show x and y axis ticks.
    
    Returns:
    - contour_obj: The contour object from the plot.
    """

    # Create a new figure/axis if none is provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(20, 15), subplot_kw={'projection': ccrs.Robinson()})
        ax.set_global()
  
    # Determine significance mask (where p-values are less than 0.05)
    # significance_mask = p_values < 0.05
    insignificance_mask = p_values >= 0.10
    
    # Plotting
    # contour_obj = ax.pcolormesh(lons, lats, trend_data,  cmap='RdBu_r',vmin=-5.0, vmax=5.0, transform=ccrs.PlateCarree(central_longitude=180), shading='auto')
    contour_obj = ax.contourf(lons, lats, trend_data, levels=levels, cmap=cmap, extend=extend,transform=ccrs.PlateCarree(central_longitude=0))

    # Plot significance masks with different hatches
    ax.contourf(lons, lats, insignificance_mask, levels=[0, 0.10, 1.0],hatches=[None,'///'], colors='none', transform=ccrs.PlateCarree())

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                      color='gray', alpha=0.35, linestyle='--')

    # Disable labels on the top and right of the plot
    gl.top_labels = False
    gl.right_labels = False

    # Enable labels on the bottom and left of the plot
    gl.bottom_labels = show_xticks
    gl.left_labels = show_yticks
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 14}
    gl.ylabel_style = {'size': 14}
    
    if show_xticks:
        gl.bottom_labels = True
    if show_yticks:
        gl.left_labels = True
    
    ax.set_title(title, loc='center', fontsize=18, pad=5.0)

    return contour_obj


In [ ]:
# define an asymmetric colormap
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.colors import BoundaryNorm
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

intervals = [-0.2, -0.15, -0.1, -0.05, 0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.9, 1.1, 1.3]

# Normalizing the intervals to [0, 1]
min_interval = min(intervals)
max_interval = max(intervals)
normalized_intervals = [(val - min_interval) / (max_interval - min_interval) for val in intervals]

# cmap = mcolors.ListedColormap(palettable.scientific.diverging.Vik_20.mpl_colors)
cmap=mcolors.ListedColormap(palettable.cmocean.diverging.Balance_20.mpl_colors)

# Define the colors at each interval
colors = [(0.00784313725490196, 0.2, 0.4627450980392157, 1.0),
    (0.00784313725490196, 0.2, 0.4627450980392157, 1.0),
    (0.023529411764705882, 0.32941176470588235, 0.5450980392156862, 1.0),
    (0.023529411764705882, 0.32941176470588235, 0.5450980392156862, 1.0),
    (1.0, 1.0, 1.0, 1.0),
    (1.0, 0.9, 0.98, 1.0),
    (1.0, 0.8, 0.5, 1.0),
    (1.0, 0.803921568627451, 0.607843137254902, 1.0), 
    (1.0, 0.6000000000000001, 0.20000000000000018, 1.0),
    (1.0, 0.4039215686274509, 0.0, 1.0),
    (0.8999999999999999, 0.19999999999999996, 0.0, 1.0),
    (0.7470588235294118, 0.0, 0.0, 1.0), 
    (0.6000000000000001, 0.0, 0.0, 1.0),
    (0.44705882352941173, 0.0, 0.0, 1.0),
    (0.30000000000000004, 0.0, 0.0, 1.0),
    (0.14705882352941177, 0.0, 0.0, 1.0),
    (0.0, 0.0, 0.0, 1.0)]

# Creating a list of tuples with normalized positions and corresponding colors
color_list = list(zip(normalized_intervals, colors))

# Create the colormap
custom_cmap = LinearSegmentedColormap.from_list('my_custom_cmap', color_list)

# Create a normalization
norm = Normalize(vmin=min_interval, vmax=max_interval)

In [ ]:
# put the data into a dictionary
# 1950-2022 raw trend
# trend_data = HadCRUT_annual_trend*10.0
# forced trend
forced_trend_data = xr.Dataset({'MMEM': HadCRUT_annual_forced_trend*10.0, 
                    'CanESM5': HadCRUT_annual_forced_trend_CanESM5*10.0,
                    'IPSL': HadCRUT_annual_forced_trend_IPSL*10.0,
                    'EC_Earth': HadCRUT_annual_forced_trend_EC_Earth*10.0,
                    'ACCESS': HadCRUT_annual_forced_trend_ACCESS*10.0,
                    'MPI_ESM': HadCRUT_annual_forced_trend_MPI_ESM*10.0,
                    'MIROC6': HadCRUT_annual_forced_trend_MIROC6*10.0}, 
                    coords={'lat': HadCRUT5['lat'], 'lon': HadCRUT5['lon']})

forced_p_val_data = xr.Dataset({'MMEM': HadCRUT_annual_forced_p_value,
                    'CanESM5': HadCRUT_annual_forced_p_value_CanESM5,
                    'IPSL': HadCRUT_annual_forced_p_value_IPSL,
                    'EC_Earth': HadCRUT_annual_forced_p_value_EC_Earth,
                    'ACCESS': HadCRUT_annual_forced_p_value_ACCESS,
                    'MPI_ESM': HadCRUT_annual_forced_p_value_MPI_ESM,
                    'MIROC6': HadCRUT_annual_forced_p_value_MIROC6},
                    coords={'lat': HadCRUT5['lat'], 'lon': HadCRUT5['lon']})

# unforced trend
unforced_trend_data = xr.Dataset({'MMEM': HadCRUT_annual_residual_trend*10.0,
                        'CanESM5': HadCRUT_annual_residual_trend_CanESM5*10.0,
                        'IPSL': HadCRUT_annual_residual_trend_IPSL*10.0,
                        'EC_Earth': HadCRUT_annual_residual_trend_EC_Earth*10.0,
                        'ACCESS': HadCRUT_annual_residual_trend_ACCESS*10.0,
                        'MPI_ESM': HadCRUT_annual_residual_trend_MPI_ESM*10.0,
                        'MIROC6': HadCRUT_annual_residual_trend_MIROC6*10.0},
                        coords={'lat': HadCRUT5['lat'], 'lon': HadCRUT5['lon']})

unforced_p_val_data = xr.Dataset({'MMEM': HadCRUT_annual_residual_p_value,
                    'CanESM5': HadCRUT_annual_residual_p_value_CanESM5,
                    'IPSL': HadCRUT_annual_residual_p_value_IPSL,
                    'EC_Earth': HadCRUT_annual_residual_p_value_EC_Earth,
                    'ACCESS': HadCRUT_annual_residual_p_value_ACCESS,
                    'MPI_ESM': HadCRUT_annual_residual_p_value_MPI_ESM,
                    'MIROC6': HadCRUT_annual_residual_p_value_MIROC6},
                    coords={'lat': HadCRUT5['lat'], 'lon': HadCRUT5['lon']})


In [ ]:
# # transform the trend data unit to degree per decade
# for i in range(len(variable_name)):
#     # forced_trend_annual_da[variable_name[i]] = forced_trend_annual_da[variable_name[i]] * 10
#     internal_trend_annual_da[variable_name[i]] = internal_trend_annual_da[variable_name[i]] * 10

In [ ]:
# # check the min and max value of the trend
# for i in range(len(variable_name)):
#     # print(variable_name[i], forced_trend_annual_da[variable_name[i]].min().values, forced_trend_annual_da[variable_name[i]].max().values)
#     print(variable_name[i], internal_trend_annual_da[variable_name[i]].min().values, internal_trend_annual_da[variable_name[i]].max().values)

In [ ]:
# check the min and max value of the trend
# print(trend_data.min().values, trend_data.max().values)

In [ ]:
print(unforced_trend_data.min().values, unforced_trend_data.max().values)

In [ ]:
# Plotting
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import numpy as np
import cartopy.util as cutil
import matplotlib.colors as mcolors
import palettable

lat = HadCRUT5.lat
lon = HadCRUT5.lon
lat, lon 

variable_name = ["MMEM", "CanESM5", "IPSL", "EC_Earth", "ACCESS", "MPI_ESM", "MIROC6"]

# titles = ["Forced.wrt.MMEM", "Forced.wrt.CanESM5", "Forced.wrt.IPSL-CM6A-LR", 
        #   "Forced.wrt.EC-Earth3", "Forced.wrt.ACCESS-ESM1.5", "Forced.wrt.MPI-ESM-LR1.2", "Forced.wrt.MIROC6"]
titles = ["Raw HadCRUT5 Trend"] + ["Forced wrt. " + model for model in variable_name] + ["Unforced wrt. " + model for model in variable_name]

titles_right = ["Unforced.wrt.MMEM", "Unforced.wrt.CanESM5", "Unforced.wrt.IPSL-CM6A-LR",
                "Unforced.wrt.EC-Earth3", "Unforced.wrt.ACCESS-ESM1.5", "Unforced.wrt.MPI-ESM-LR1.2", "Unforced.wrt.MIROC6"]

titles_label = ["a. ", "b. ", "c. ", "d. ", "e. ", "f. ", "g. ", "h. "]

# Figure arange as: 8 rows and 2 columns, with the first column for the forced trend and the second column for the unforced trend;
# The first row is for the raw HadCRUT5 trend, the following row will show the trend for each model

intervals = np.arange(-0.5, 0.55, 0.05)
# cmap = mcolors.ListedColormap(palettable.cmocean.sequential.Amp_20.mpl_colors)
cmap=mcolors.ListedColormap(palettable.cmocean.diverging.Balance_20.mpl_colors)

fig  = plt.figure(figsize=(15,30))
gs = gridspec.GridSpec(8, 2, figure=fig, wspace=0.01, hspace=0.15)
extend ='both'

ax = fig.add_subplot(gs[0, 0], projection=ccrs.Robinson(180))
trend_data = HadCRUT_annual_trend*10.0
p_values = HadCRUT_annual_p_value
trend_data_with_cyclic, lons_cyclic = cutil.add_cyclic_point(trend_data, coord=lon)
p_values_with_cyclic, lons_cyclic = cutil.add_cyclic_point(p_values, coord=lon)
contour_obj = plot_trend_with_significance(trend_data_with_cyclic, lat, lons_cyclic, p_values_with_cyclic, 
                                           levels=intervals, extend=extend,
                                           cmap='twilight_shifted', title=titles[0], ax=ax, show_xticks=False, show_yticks=True)
#add title for the first row
ax.text(-0.15, 1.1, titles_label[0], va='bottom', ha='center', rotation='horizontal', fontsize=18,
        weight='bold', transform=ax.transAxes)
axes = {}
for i in range(1,8):
    for j in range(2):
        is_left = j == 0
        is_bottom_row = i >= 7
        
        ax = fig.add_subplot(gs[i, j], projection=ccrs.Robinson(180))
        ax.set_global()
        axes[(i, j)] = ax
        if j == 0:
            # Add cyclic points
            print(variable_name[i-1])
            trend_data_var = forced_trend_data[variable_name[i-1]]
            p_values_var = forced_p_val_data[variable_name[i-1]]
            trend_with_cyclic, lons_cyclic = cutil.add_cyclic_point(trend_data_var, coord=lon)
            p_values_with_cyclic, lons_cyclic = cutil.add_cyclic_point(p_values_var, coord=lon)
            
            contour_obj1 = plot_trend_with_significance(trend_with_cyclic, lat, lons_cyclic, p_values_with_cyclic, 
                                                       levels=intervals, extend=extend,
                                                       cmap='twilight_shifted', title=titles[i], ax=ax, show_xticks = is_bottom_row, show_yticks = is_left)
        else:
            trend_data_var = unforced_trend_data[variable_name[i-1]]
            p_values_var =unforced_p_val_data[variable_name[i-1]]
            trend_with_cyclic, lons_cyclic = cutil.add_cyclic_point(trend_data_var, coord=lon)
            p_values_with_cyclic, lons_cyclic = cutil.add_cyclic_point(p_values_var, coord=lon)
            
            contour_obj2 = plot_trend_with_significance(trend_with_cyclic, lat, lons_cyclic, p_values_with_cyclic, levels=intervals,
                                                        extend=extend, cmap='twilight_shifted', title=titles[i+7], 
                                                        ax=ax, show_xticks = is_bottom_row, show_yticks = is_left)

# add the title for each row and column
# axes[0, 0].text(-0.15, 1.1, titles_label[0], va='bottom', ha='center', rotation='horizontal', fontsize=22, 
#                     weight='bold', transform=axes[0, 0].transAxes)

for i in range(1,8):
    axes[i, 0].text(-0.15, 1.1, titles_label[i], va='bottom', ha='center', rotation='horizontal', fontsize=18, 
                    weight='bold', transform=axes[i, 0].transAxes)

# add colorbar for each row [degc/60yr, 30yr, 10yr], horizontal colorbar
cbar_ax = fig.add_axes([0.25, 0.08, 0.5, 0.01])
cbar = fig.colorbar(contour_obj, cax=cbar_ax, orientation='horizontal', extend=extend)
# cbar.set_label('SAT trend ($^\circ$C/60yr)', fontsize=14)
fig.text(0.5, 0.06,'Annual SAT trend ($^\circ$C/decade)',fontsize=16, ha='center', va='bottom')
cbar.ax.tick_params(labelsize=14)

# plt.figure(constrained_layout=True)
# fig.savefig('FigureS1-1950-2022-HadCRUT5-Trend-separation.png', dpi=300, bbox_inches='tight')
fig.savefig('FigureS1-1950-2022-HadCRUT5-Trend-separation(NaNprocess)-revised.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
client.close()
scluster.close()